# Overview

## Organisation of documentation
All the documentation is presented as Jupyter notebooks with naming convention `Documentation_X_Y.ipynb`, with `X` and `Y` being integers as follows:
* `X=0`: Basics and overview
* `X=1`: Demonstration of basic functionality
* `X=2`: Example use cases

## Purpose of code
The purpose of the Grain Boundary ToolKit (GBTK) is to provide some general purpose tools for the generatation of simulation supercells containing grain boundaries.

## Organisation of code
The GBTK code is modular with a reasonably straightforward heirarchy: 

* `lattice.py`: contains descriptions of different lattice types.
* `csl.py`: contains a description of a coincident site lattice (CSL). This includes the axis-angle degrees of freedom of a grain boundary description by definition. It contains useful tools for cataloguing grain boundaries and for calculating the DSC lattice.
* `grainboundary.py`: contains a description of a grain boundary at the level of the five macroscopic degrees of freedom. If the boundary is a CSL boundary, then a csl must be defined and then the grain boundary object contains the two remaining degrees of freedom defining the boundary plane.
* `gbsupercell.py`: contains descriptions of simulation supercells for grain boundary systems. Microscopic degrees of freedom for the grain boundary are dealt with here. It also contains methods for writing out files.

All the above modules also contain functionality for visualising the output with Plotly via Jupyter notebooks. They will also generate informative output (partly for debugging purposes) if requested. The remainder of the documentation will make use of this.

In addition to the above, there are several further "helper" modules:

* `spatialsearch.py`: holds a description of a search space to speed up searches for e.g. the csl basis of grain boundaries.
* `crystaltools.py`: contains functions to perform some common tasks such as decomposing a vector in a given basis or wrapping position and vectors to periodic cells. It also contains some commonly needed functionality for the visualisation functions in the main modules.

## Specific notes on modules
Below are some notes specific to the different modules, with the aim of making them more accessible and reducing the chances of them failing in use.

### `lattice.py`
A lattice is specified by its `lattice_type`. It has `cell_vectors` specified in the cartesian basis, which in general define the edges of a parallelipiped. The array `basis_coords` specifies the location of the atoms in the unit cell in the basis defined by the `cell_vectors`. A further array of `atom_types` contains integer identifiers for the atom types in the basis. These need not correspond to atomic species. Indeed, it would be wise to give non-equivalent atoms of the same species different type labels, for reasons highlighted in the notes below. Note the following:
* The (0,0,0) point of the lattice unit cell must be a point of inversion symmetry if you wish for a pair of equivalent boundaries in a periodic supercell.
* Only the (0,0,0) point is considered in detecting the CSL. If your unit cell contains multiple equivalent atoms then this will lead to a larger csl cell than necessary. This can be avoided by moving to a primitive unit cell if you have a strong need to minimise numbers of atoms.
* All `atom_type==1` atoms in the basis will be used in determining the DSC lattice. This may influence your approach to choosing the `atom_types` of non-equivalent atoms of the same species in a unit cell.
* The cell will be scaled by a chosen lattice parameter when used. Hence the primary lattice vector should be set to unit length here.

### `csl.py`
The approach taken to setting up a csl is to specify an axis-angle combination. The angle can be given via a pair of integers for some crystal systems. The module calculates details of the csl unit cell and will also generate the DSC lattice. A key feature of the `csl.py` module is the functionality to write out catalogues of grain boundaries with a given axis, including details of $\Sigma$ values and csl unit cell vectors.  These catalogues can be used to select appropriate sets of boundaries for further study.

### `grainboundary.py`
This module deals with the full five macroscopic degrees of freedom of grain boundary geometry. Given a grain boundary plane specified by a sequence of miller-like indices in the csl basis, the module will return details of the boundary plane normals and the minimal cell required to contain the crystal with a boundary of this geometry. Importantly, for a chosen axis-angle combination, it is possible to write out a catalogue of grain boundary cells with a variety of grain boundary planes, including details of the boundary plane normals and the number of atoms in the cells. Again, the purpose of these catalogues is to allow selection of particular boundaries (perhaps all those within a given size limit).

### `supercell.py`
A supercell is the object that we ultimately want to simulate. This may be a bicrystal cell or a spherical cluster. This module takes a specified boundary and generates a set of atom positions to represent it. This module handles all the microscopic degrees of freedom of the grain boundary, including relative grain translations, grain boundary plane shifts and the addition of excess volume at the boundary. It also deals with any additional simulation constraints, such as vacuum at the ends of the supercell, or fixing of atom positions.

The approach taken to the microscopic degrees of freedom associated with relative grain translations is one of the following. Either:
* Atoms of one grain are translated with respect to the other by a vector *in the grain boundary plane* specified as fractions of the two in-plane csl unit cell vectors. This mode of translation corresponds to the classic gamma surface exploration.
or:
* Atoms of one grain are translated with respect to the other by fractions of the three DSC unit cell vectors. In addition to this, the nominal grain boundary planes are moved by a fraction of the maximum out of plane distance contained within the DSC unit cell (projection of the DSC unit cell diagonal onto the grain boundary plane normal). This gives four degrees of freedom in total and will allow for the variation of grain boundary termination planes in crystals with a motif.

*Important*: The construction of periodic bicrystal supercells is implemented in such a way as to preserve the equivalence of the two grain boundaries. This is achieved by making all changes (deletion of atoms, translation of grains, expansion of boundaries, etc.) in such a way that they respect inversion symmetry about the centre of the supercell. This will only work when the `lattice` object is specified in such a way that the point (0,0,0) is a centre of inversion symmetry.

